<a href="https://colab.research.google.com/github/mike66612/FOC-basic/blob/main/%E2%80%9CHF_Weight_Image_Download_ipynb%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 权重与图片下载配置

在第 2 个代码单元格填写 `HF_WEIGHT_URL`，然后从上往下运行。链接为空时会跳过权重下载。

图片链接已经预填。下载代码只保存文件，不打开、解析或显示图片，也不加载权重。此笔记本仅准备输入文件；模型加载和推理需要对应模型的实现。

默认保存到 `/content/hf_image_work`。Colab 临时运行环境释放后文件可能丢失，需要保留时自行下载或修改保存目录。受限权重可在 Colab 的“密钥”面板中添加 `HF_TOKEN`。

In [ ]:
# @title 1. 安装下载依赖
%pip -q install requests gdown

In [ ]:
# @title 2. 配置下载链接
from pathlib import Path

HF_WEIGHT_URL = "https://huggingface.co/mohss/bbbbbb11/resolve/main/GodPussy1%20v4.safetensors" # @param {type:"string"}
IMAGE_URL = "https://drive.google.com/file/d/1o3qxGqv2hZSTPiXNJvkUG6r3qaqZGwpb/view?usp=drivesdk"
WORK_DIR = Path("/content/hf_image_work")
WEIGHT_DIR = WORK_DIR / "weights"
INPUT_IMAGE_PATH = WORK_DIR / "input_image"
OUTPUT_DIR = WORK_DIR / "outputs"
for directory in (WORK_DIR, WEIGHT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
WEIGHT_PATH = None
print("配置完成。未打开图片或权重。")

配置完成。未打开图片或权重。


In [ ]:
# @title 3. 下载权重（链接为空时跳过）
from urllib.parse import urlparse, unquote
import requests

WEIGHT_PATH = None
url = HF_WEIGHT_URL.strip()
if not url:
    print("HF_WEIGHT_URL 为空：已跳过权重下载。填写后重跑此单元格即可。")
else:
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname != "huggingface.co" or "/resolve/" not in parsed.path:
        raise ValueError("请填写 Hugging Face 文件下载链接：https://huggingface.co/仓库/resolve/版本/文件名")
    filename = Path(unquote(parsed.path)).name
    if not filename or filename in (".", ".."):
        raise ValueError("下载链接缺少文件名。")
    destination = WEIGHT_DIR / filename
    temporary = destination.with_name(destination.name + ".part")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    headers = {"Authorization": "Bearer " + token} if token else {}
    try:
        with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
            if response.status_code != 200:
                raise RuntimeError(f"权重下载失败（HTTP {response.status_code}）。请检查链接和访问权限。")
            if "text/html" in response.headers.get("Content-Type", "").lower():
                raise RuntimeError("返回的是网页，请改用文件的 resolve 下载链接。")
            with temporary.open("wb") as output:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        output.write(chunk)
        if temporary.stat().st_size == 0:
            raise RuntimeError("下载结果为空。")
        temporary.replace(destination)
        WEIGHT_PATH = destination
        print("权重下载完成，路径：", str(WEIGHT_PATH))
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    finally:
        headers.clear()
        token = None

权重下载完成，路径： /content/hf_image_work/weights/GodPussy1 v4.safetensors


In [ ]:
%pip -q install google-api-python-client

In [ ]:
from pathlib import Path
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import google.auth

# 1. Google Drive 文件 ID
FILE_ID = "1o3qxGqv2hZSTPiXNJvkUG6r3qaqZGwpb"

# 2. 保存路径
INPUT_IMAGE_PATH = Path("/content/input_image.jpg")

# 3. 授权 Google 账号（需有该文件访问权限）
auth.authenticate_user()

creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)

service = build(
    "drive",
    "v3",
    credentials=creds,
    cache_discovery=False
)

# 4. 通过授权的 Drive API 下载文件
request = service.files().get_media(fileId=FILE_ID)

temporary_image = INPUT_IMAGE_PATH.with_name(
    INPUT_IMAGE_PATH.name + ".part"
)

try:
    with open(temporary_image, "wb") as f:
        downloader = MediaIoBaseDownload(
            f,
            request,
            chunksize=1024 * 1024
        )

        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"下载进度：{status.progress() * 100:.1f}%")

    if temporary_image.stat().st_size == 0:
        raise RuntimeError("下载文件为空")

    temporary_image.replace(INPUT_IMAGE_PATH)

    print("下载成功！")
    print("保存位置：", INPUT_IMAGE_PATH)
    print("文件大小：", INPUT_IMAGE_PATH.stat().st_size)
    print("未打开、解析或预览图片内容。")

except Exception:
    temporary_image.unlink(missing_ok=True)
    raise

下载进度：100.0%
下载成功！
保存位置： /content/input_image.jpg
文件大小： 54068
未打开、解析或预览图片内容。


In [ ]:
# @title 5. 文件准备状态
print("权重：", str(WEIGHT_PATH) if WEIGHT_PATH and WEIGHT_PATH.is_file() else "未下载，请先填写权重链接")
print("图片：", str(INPUT_IMAGE_PATH) if INPUT_IMAGE_PATH.is_file() else "未下载")
print("输出目录：", str(OUTPUT_DIR))
print("此处只检查文件是否存在，不读取图片或权重内容。")

权重： /content/hf_image_work/weights/GodPussy1 v4.safetensors
图片： /content/input_image.jpg
输出目录： /content/hf_image_work/outputs
此处只检查文件是否存在，不读取图片或权重内容。


In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler
from PIL import Image
import requests
from io import BytesIO

# ====================== 配置区域 ======================
base_model_id = "runwayml/stable-diffusion-inpainting"   # 官方 inpainting 底模
# 或者你喜欢的底模（LazyMix+ / RealHotMix 等），需要自己转成 inpainting 版本

lora_path = "/content/GodPussy1_v4_Inpainting.safetensors"   # 你的 LoRA 路径
lora_weight = 0.75                                          # 推荐 0.5 \~ 1.0

prompt = """
highly detailed realistic pussy, perfect anatomy, detailed labia, natural skin texture,
wet, glossy, soft lighting, photorealistic, 8k
"""

negative_prompt = """
fingers, hands, deformed, extra limbs, blurry, low quality, cartoon, anime,
text, watermark, bad anatomy, double butthole
"""

# 图片和遮罩（必须是 512x768 效果最好）
init_image = Image.open("your_image.png").convert("RGB").resize((512, 768))
mask_image = Image.open("your_mask.png").convert("RGB").resize((512, 768))  # 白色=要重绘区域
# =====================================================

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    safety_checker=None,          # 关闭安全检查
    requires_safety_checker=False
).to("cuda")

# 换成更快的调度器
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# 加载 LoRA
pipe.load_lora_weights(lora_path)
pipe.fuse_lora(lora_scale=lora_weight)

# 生成
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    mask_image=mask_image,
    height=768,
    width=512,
    num_inference_steps=28,
    guidance_scale=7.0,
    strength=0.55,                # 对应 denoising strength，0.4\~0.7 比较常用
).images[0]

result.save("godpussy_result.png")
print("生成完成！")

FileNotFoundError: [Errno 2] No such file or directory: 'your_image.png'

In [ ]:
import cv2
import numpy as np
from pathlib import Path

# ================= 配置 =================
IMAGE_PATH = Path("/content/input_image.jpg")
MASK_PATH = Path("/content/input_mask.png")

# 越大检测越宽松（建议 20~40）
DETECT_PERCENTILE = 30

# 遮罩向外扩张像素数
DILATE_PIXELS = 7

# 最小候选区域占图片面积比例
MIN_AREA_RATIO = 0.003

# 可选：只在指定区域检测 (x1, y1, x2, y2)
# None 表示全图检测
ROI = None

# ================= 读取图片 =================
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"图片不存在: {IMAGE_PATH}")

img = cv2.imread(str(IMAGE_PATH))
if img is None:
    raise RuntimeError("无法读取图片")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = gray.astype(np.float32) / 255.0

h, w = gray.shape

# ================= 1. Laplacian =================
# 计算高频细节
smooth = cv2.GaussianBlur(gray, (3, 3), 0)

lap = cv2.Laplacian(
    smooth, cv2.CV_32F, ksize=3
)

lap_energy = cv2.boxFilter(
    lap ** 2,
    -1,
    (21, 21)
)

# ================= 2. Sobel =================
# 检测局部边缘
gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)

sobel_energy = cv2.boxFilter(
    gx ** 2 + gy ** 2,
    -1,
    (21, 21)
)

# ================= 3. 局部方差 =================
mean = cv2.boxFilter(
    gray, -1, (21, 21)
)

mean_sq = cv2.boxFilter(
    gray ** 2, -1, (21, 21)
)

variance = np.maximum(
    mean_sq - mean ** 2, 0
)

# ================= 4. 特征融合 =================
def normalize(x):
    scale = np.percentile(x, 95)
    return np.clip(x / (scale + 1e-8), 0, 1)

score = (
    0.45 * normalize(lap_energy)
    + 0.35 * normalize(sobel_energy)
    + 0.20 * normalize(variance)
)

# 分数越低，越可能是模糊或低纹理区域
threshold = np.percentile(
    score, DETECT_PERCENTILE
)

candidate = (score < threshold).astype(np.uint8) * 255

# ================= 5. 可选 ROI 限制 =================
if ROI is not None:
    x1, y1, x2, y2 = ROI
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    roi_mask[y1:y2, x1:x2] = 255
    candidate = cv2.bitwise_and(candidate, roi_mask)

# ================= 6. 去噪与区域合并 =================
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (11, 11)
)

candidate = cv2.morphologyEx(
    candidate, cv2.MORPH_CLOSE, kernel
)

candidate = cv2.morphologyEx(
    candidate, cv2.MORPH_OPEN,
    np.ones((3, 3), np.uint8)
)

# 删除过小区域
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    candidate, connectivity=8
)

mask = np.zeros((h, w), dtype=np.uint8)
min_area = h * w * MIN_AREA_RATIO

for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]
    if area >= min_area:
        mask[labels == i] = 255

# ================= 7. 遮罩膨胀 =================
if DILATE_PIXELS > 0:
    size = 2 * DILATE_PIXELS + 1
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (size, size)
    )
    mask = cv2.dilate(mask, kernel, iterations=1)

# ================= 8. 保存遮罩 =================
cv2.imwrite(str(MASK_PATH), mask)

ratio = np.count_nonzero(mask) / mask.size * 100

print("自动检测完成")
print("遮罩保存位置:", MASK_PATH)
print(f"检测遮罩面积比例: {ratio:.2f}%")
print("白色 = 需要重绘")
print("黑色 = 保留原图")
print("未显示或预览图片")

if ratio > 60:
    print("警告：检测面积过大，可能把正常平滑区域误判为模糊区域。")
elif ratio < 1:
    print("提示：检测面积较小，可能存在漏检。")

自动检测完成
遮罩保存位置: /content/input_mask.png
检测遮罩面积比例: 37.09%
白色 = 需要重绘
黑色 = 保留原图
未显示或预览图片


In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline, DPMSolverMultistepScheduler
from PIL import Image, ImageFilter, ImageOps
import numpy as np
import os

# ====================== 配置区域 ======================
base_model_id = "runwayml/stable-diffusion-inpainting"

# 你真实下载好的权重路径（带空格）
lora_path = "/content/hf_image_work/weights/GodPussy1 v4.safetensors"
lora_weight = 0.8          # 可以 0.6\~1.0 之间试

prompt = """
highly detailed realistic pussy, perfect anatomy, detailed labia, natural skin texture,
wet, glossy, soft lighting, photorealistic, 8k, sharp focus
"""

negative_prompt = """
fingers, hands, deformed, extra limbs, blurry, low quality, cartoon, anime,
text, watermark, bad anatomy, double butthole, mosaic, pixelated, censored
"""

# 输入图
init_image = Image.open("/content/input_image.jpg").convert("RGB").resize((512, 768))

# ---------- 自动生成遮罩（针对轻微模糊/马赛克区域） ----------
def create_blur_mask(img, blur_threshold=15):
    """
    简单自动遮罩：计算局部清晰度，模糊区域变白（需要重绘）
    如果整张图都比较糊，会接近全白遮罩（推荐）
    """
    gray = img.convert("L")
    # 用拉普拉斯方差近似清晰度
    arr = np.array(gray, dtype=np.float32)
    # 简单边缘/方差图
    from scipy.ndimage import uniform_filter
    mean = uniform_filter(arr, size=15)
    sqr_mean = uniform_filter(arr**2, size=15)
    variance = sqr_mean - mean**2

    # 方差低 = 模糊 → 白色（重绘）
    mask_arr = (variance < blur_threshold).astype(np.uint8) * 255
    mask = Image.fromarray(mask_arr).convert("RGB")

    # 稍微膨胀一点，把边缘也盖住
    mask = mask.filter(ImageFilter.MaxFilter(5))
    return mask

# 生成遮罩（如果自动检测效果不好，下面一行改成全白遮罩更稳）
mask_image = create_blur_mask(init_image, blur_threshold=12)

# 如果想强制整张图重绘（推荐先试这个），取消下面注释：
# mask_image = Image.new("RGB", (512, 768), (255, 255, 255))

mask_image.save("/content/auto_mask.png")   # 保存看看遮罩长什么样
print("遮罩已自动生成并保存为 /content/auto_mask.png")

# =====================================================

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# 加载 LoRA
pipe.load_lora_weights(lora_path)
pipe.fuse_lora(lora_scale=lora_weight)

# 生成
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    mask_image=mask_image,
    height=768,
    width=512,
    num_inference_steps=30,
    guidance_scale=7.5,
    strength=0.65,          # 模糊比较重时可以开到 0.7\~0.8
).images[0]

result.save("/content/godpussy_result.png")
print("生成完成！结果保存在 /content/godpussy_result.png")

遮罩已自动生成并保存为 /content/auto_mask.png


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


AssertionError: Torch not compiled with CUDA enabled